# 365 Probabilidades - Dia #138
## Qual a probabilidade de um exame positivo estar errado?

**Tipo:** Raciocínio
**Data de publicação:** 2026-10-29
**Ferramenta:** Python
**Decisão analisada:** Um exame deu positivo. O que isso quer dizer sobre a chance de eu ter a doença?
**Hashtag:** #365Probabilidades #Dia138

---

### 📖 A História

Antes de ler, faça a conta de cabeça.

Uma doença atinge 1 pessoa em cada 1.000. O exame que detecta essa doença tem 5% de
falso positivo, ou seja, de cada 100 pessoas sadias, 5 testam positivo. Uma pessoa sem
nenhum sintoma faz o exame e o resultado vem positivo. Qual a chance de ela ter a
doença?

Guarde o seu número. Essa pergunta foi feita a médicos de hospitais de Boston duas vezes,
em 1978 e em 2014. A resposta mais comum foi 95%. A resposta certa é 2%.

---

### 📚 O Conceito: a taxa-base

Um exame positivo responde a uma pergunta: se a pessoa tem a doença, o exame acusa? A
pessoa que recebe o resultado quer saber outra coisa: se o exame acusou, ela tem a
doença? As duas perguntas só têm a mesma resposta quando a doença é comum.

Quando a doença é rara, a maior parte das pessoas que fazem o exame é sadia, e 5% de
muitos sadios dá mais positivos do que 100% de poucos doentes. A proporção de doentes
antes do exame se chama taxa-base, e é ela que o raciocínio costuma esquecer. O teorema
de Bayes é só a conta que coloca a taxa-base de volta.

O jeito mais fácil de fazer essa conta não é com porcentagens. É com pessoas. De 1.000
pessoas, 1 tem a doença e o exame a encontra. Das 999 sadias, 5% testam positivo, o que
dá cerca de 50 pessoas. São 51 positivos, e só 1 é verdadeiro.

---

### 🧮 O Modelo

Cinco camadas. A fonte dá as respostas dos médicos. As contas são nossas.

1. **A conta do estudo.** Em frequências naturais e pela fórmula de Bayes, com a
   sensibilidade de 100% que a pergunta supõe e com 90%, para mostrar o que muda.
2. **Os médicos.** Quantos acertaram em 2014, quantos em 1978, os dois juntos, e qual foi
   a resposta mais comum. Intervalos de Jeffreys.
3. **A curva.** A chance de o positivo estar errado em função da taxa-base, para o exame
   do estudo e para exames mais específicos. O ponto em que o positivo passa a ter mais
   chance de estar certo do que errado.
4. **Um grupo de mil.** Simulação de 200 mil grupos de 1.000 pessoas rastreadas, para
   ver quantos positivos aparecem e em quantos grupos todos os positivos são falsos.
5. **Quatro exames de verdade.** O que 553 profissionais de saúde estimaram para um
   positivo em quatro cenários clínicos, contra a faixa que a evidência sustenta.

**Fonte central:**
- Manrai, A. K., Bhatia, G., Strymish, J., Kohane, I. S. & Jain, S. H. (2014).
  "Medicine's uncomfortable relationship with math: calculating positive predictive
  value". *JAMA Internal Medicine* 174(6), 991-993. DOI 10.1001/jamainternmed.2014.1059.
  **N = 61** (24 médicos titulares, 26 residentes, 10 estudantes de medicina e 1 médico
  aposentado), em hospitais de Boston. Pergunta: prevalência de 1 em 1.000, falso
  positivo de 5%, sem sintomas. Resposta certa 1,96% (aceitas "2%", "1,96%" e "<2%").
  **Acertaram 14 de 61 (23%).** Resposta mais comum: 95%, dada por 27 de 61. Respostas
  de 0,005% a 96%. Mediana: 66%. Réplica de Casscells, Schoenberger & Graboys (1978),
  *New England Journal of Medicine* 299(18), 999-1001, N = 60 (11 acertos, 27 responderam
  95%); a diferença entre os dois estudos não é significativa.

**Fonte de apoio:**
- Morgan, D. J., Pineles, L., Owczarzak, J. et al. (2021). "Accuracy of practitioner
  estimates of probability of diagnosis before and after testing". *JAMA Internal
  Medicine* 181(6), 747-755. DOI 10.1001/jamainternmed.2021.0269. 723 profissionais
  convidados, 585 responderam, **553 completaram**. Estimativa mediana da chance de doença
  depois de um exame positivo, contra a faixa da evidência: pneumonia com radiografia
  95% (evidência 46% a 65%); câncer de mama com mamografia de rastreio 50% (3% a 9%);
  isquemia cardíaca com teste de esforço 70% (2% a 11%); infecção urinária com urocultura
  80% (0% a 8,3%).

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. A medida central é a
proporção de acertos numa pergunta com resposta certa, e não autorrelato. As estimativas
de Morgan et al. também não recebem o fator: o erro de estimativa é o próprio objeto.

**Ponte interna:** #062 (o mais novo da sala e o diagnóstico), outro mecanismo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 200_000
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = ('Fonte: Manrai et al. (2014), JAMA Internal Medicine 174(6), N = 61 · Casscells et al. (1978), NEJM · '
          'cálculo próprio | #365Probabilidades')

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

def vpp(prevalencia, sensibilidade, especificidade):
    # Bayes: P(doente | positivo)
    vp = sensibilidade * prevalencia
    fp = (1 - especificidade) * (1 - prevalencia)
    return vp / (vp + fp)

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---

# A pergunta feita aos médicos (Casscells 1978; Manrai 2014)
PREVALENCIA = 1 / 1000          # 1 pessoa em 1.000 tem a doença
FALSO_POSITIVO = 0.05           # 5% dos sadios testam positivo
SENSIBILIDADE = 1.00            # a pergunta não fala em falso negativo; supõe-se que o exame acha todos os doentes

# Manrai et al. 2014 · N = 61
N_2014 = 61
ACERTOS_2014 = 14
RESP_95_2014 = 27               # responderam "95%"
MEDIANA_2014 = 0.66

# Casscells et al. 1978 · N = 60
N_1978 = 60
ACERTOS_1978 = 11
RESP_95_1978 = 27

# Morgan et al. 2021 · N = 553 · mediana estimada depois do positivo e faixa da evidência
MORGAN = {
    'pneumonia\n(radiografia)':        (0.95, 0.46, 0.65),
    'câncer de mama\n(mamografia)':    (0.50, 0.03, 0.09),
    'isquemia cardíaca\n(teste de esforço)': (0.70, 0.02, 0.11),
    'infecção urinária\n(urocultura)': (0.80, 0.00, 0.083),
}

print("=" * 70)
print("  DADOS · A PERGUNTA E AS RESPOSTAS")
print("=" * 70)
print(f"\n  Prevalência: 1 em {mil(1 / PREVALENCIA)} · falso positivo: {vir(FALSO_POSITIVO * 100, 0)}% · sem sintomas")
print(f"\n  Manrai et al. 2014 (N = {N_2014}): acertaram {ACERTOS_2014} · responderam 95%: {RESP_95_2014} · mediana das respostas {vir(MEDIANA_2014 * 100, 0)}%")
print(f"  Casscells et al. 1978 (N = {N_1978}): acertaram {ACERTOS_1978} · responderam 95%: {RESP_95_1978}")
print(f"\n  Morgan et al. 2021 (N = 553), chance estimada de doença depois de um positivo:")
for k, (est, lo, hi) in MORGAN.items():
    print(f"    {k.replace(chr(10), ' '):<38} estimada {vir(est * 100, 0):>3}% · evidência {vir(lo * 100, 0)}% a {vir(hi * 100, 1)}%")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# CAMADA 1 · A conta do estudo
GRUPO = 1000
doentes = PREVALENCIA * GRUPO
verdadeiros = doentes * SENSIBILIDADE
falsos = (GRUPO - doentes) * FALSO_POSITIVO
p_certo = vpp(PREVALENCIA, SENSIBILIDADE, 1 - FALSO_POSITIVO)
p_errado = 1 - p_certo
p_certo_90 = vpp(PREVALENCIA, 0.90, 1 - FALSO_POSITIVO)
print("=" * 70)
print("  CAMADA 1 · A CONTA DO ESTUDO")
print("=" * 70)
print(f"\n  De {mil(GRUPO)} pessoas: {vir(doentes, 0)} doente e {vir(GRUPO - doentes, 0)} sadias")
print(f"  Positivos verdadeiros: {vir(verdadeiros, 0)} · falsos positivos: {vir(falsos, 2)} · positivos no total: {vir(verdadeiros + falsos, 2)}")
print(f"\n  P(doente | positivo) = {vir(verdadeiros, 0)} / {vir(verdadeiros + falsos, 2)} = {vir(p_certo * 100, 2)}%")
print(f"  P(positivo errado)   = {vir(p_errado * 100, 1)}%")
print(f"  Com sensibilidade de 90% em vez de 100%: P(doente | positivo) = {vir(p_certo_90 * 100, 2)}%")
print(f"\n  A resposta mais comum dos médicos foi 95%. É {vir(0.95 / p_certo, 0)} vezes a resposta certa.")

# CAMADA 2 · Os médicos
d14 = jeffreys(ACERTOS_2014, N_2014); d78 = jeffreys(ACERTOS_1978, N_1978)
d_tot = jeffreys(ACERTOS_2014 + ACERTOS_1978, N_2014 + N_1978)
d95 = jeffreys(RESP_95_2014 + RESP_95_1978, N_2014 + N_1978)
print("\n" + "=" * 70)
print("  CAMADA 2 · OS MÉDICOS")
print("=" * 70)
for nome, k, n, d in (('2014', ACERTOS_2014, N_2014, d14), ('1978', ACERTOS_1978, N_1978, d78)):
    ic = d.interval(0.95)
    print(f"\n  {nome}: acertaram {k} de {n} = {vir(k / n * 100)}%  IC 95% [{vir(ic[0] * 100)}; {vir(ic[1] * 100)}]")
ic_tot = d_tot.interval(0.95); ic95 = d95.interval(0.95)
k_tot = ACERTOS_2014 + ACERTOS_1978; n_tot = N_2014 + N_1978
print(f"\n  Os dois juntos: {k_tot} de {n_tot} = {vir(k_tot / n_tot * 100)}%  IC 95% [{vir(ic_tot[0] * 100)}; {vir(ic_tot[1] * 100)}]")
print(f"  Erraram: {vir((1 - k_tot / n_tot) * 100)}%, três em cada quatro")
print(f"  Responderam 95%: {RESP_95_2014 + RESP_95_1978} de {n_tot} = {vir((RESP_95_2014 + RESP_95_1978) / n_tot * 100)}%  [{vir(ic95[0] * 100)}; {vir(ic95[1] * 100)}]")
p_diff = stats.fisher_exact([[ACERTOS_2014, N_2014 - ACERTOS_2014], [ACERTOS_1978, N_1978 - ACERTOS_1978]])[1]
print(f"  Diferença entre 1978 e 2014: p = {vir(p_diff, 2)} (Fisher). Em 36 anos, nada mudou.")
print(f"  Mediana das respostas em 2014: {vir(MEDIANA_2014 * 100, 0)}%, contra {vir(p_certo * 100, 0)}% certo")

# CAMADA 3 · A curva
prev = np.logspace(-4, np.log10(0.6), 500)
curva_95 = 1 - vpp(prev, SENSIBILIDADE, 0.95)
curva_99 = 1 - vpp(prev, SENSIBILIDADE, 0.99)
curva_999 = 1 - vpp(prev, SENSIBILIDADE, 0.999)
virada = FALSO_POSITIVO / (FALSO_POSITIVO + SENSIBILIDADE)      # prevalência em que P(certo) = P(errado)
print("\n" + "=" * 70)
print("  CAMADA 3 · A CURVA DE BAYES")
print("=" * 70)
print(f"\n  Exame do estudo (especificidade 95%): P(positivo errado) por taxa-base")
for p in (0.001, 0.01, 0.05, 0.10, 0.30):
    print(f"    taxa-base {vir(p * 100, 1):>5}%  →  {vir((1 - vpp(p, SENSIBILIDADE, 0.95)) * 100, 1):>5}% dos positivos errados")
print(f"\n  O positivo passa a ter mais chance de estar certo do que errado quando a taxa-base passa de {vir(virada * 100, 1)}%.")
print(f"  Com 1 em 1.000, mesmo um exame com 99,9% de especificidade erra em {vir((1 - vpp(0.001, 1, 0.999)) * 100, 0)}% dos positivos.")

# CAMADA 4 · Um grupo de mil
n_doentes = rng.binomial(GRUPO, PREVALENCIA, SORTEIOS)
vp_sim = rng.binomial(n_doentes, SENSIBILIDADE)
fp_sim = rng.binomial(GRUPO - n_doentes, FALSO_POSITIVO)
positivos = vp_sim + fp_sim
todos_falsos = (vp_sim == 0) & (positivos > 0)
print("\n" + "=" * 70)
print(f"  CAMADA 4 · {mil(SORTEIOS)} GRUPOS DE {mil(GRUPO)} PESSOAS RASTREADAS")
print("=" * 70)
print(f"\n  Positivos por grupo: média {vir(positivos.mean())} · de {np.percentile(positivos, 2.5):.0f} a {np.percentile(positivos, 97.5):.0f} em 95% dos grupos")
print(f"  Doentes por grupo: média {vir(n_doentes.mean(), 2)} · grupos sem nenhum doente: {vir((n_doentes == 0).mean() * 100)}%")
print(f"  Grupos em que TODOS os positivos são falsos: {vir(todos_falsos.mean() * 100)}%")
print(f"  Proporção de positivos errados, somando todos os grupos: {vir(fp_sim.sum() / positivos.sum() * 100, 1)}%")

# CAMADA 5 · Quatro exames de verdade
print("\n" + "=" * 70)
print("  CAMADA 5 · QUATRO EXAMES DE VERDADE (MORGAN ET AL. 2021, N = 553)")
print("=" * 70)
print(f"\n  {'cenário':<36} {'estimado':>9} {'evidência':>14} {'excesso':>9}")
for k, (est, lo, hi) in MORGAN.items():
    meio = (lo + hi) / 2
    print(f"  {k.replace(chr(10), ' '):<36} {vir(est * 100, 0):>8}% {vir(lo * 100, 0):>6}% a {vir(hi * 100, 1):>4}% {vir(est / meio, 1):>7}×")
print("\n  Em três dos quatro cenários a estimativa fica de 8 a 19 vezes acima da evidência.")
print("  O cenário em que os profissionais chegam perto é o da doença comum entre quem faz o exame (pneumonia).")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · Mil pessoas
fig1, ax1 = plt.subplots(figsize=(12, 8))
lado = 40
cores_grid = np.full(GRUPO, 0)
cores_grid[:int(round(falsos))] = 1
cores_grid[int(round(falsos))] = 2
mapa = {0: '#e6e4df', 1: VERMELHO, 2: VERDE}
for i in range(GRUPO):
    r, c = divmod(i, lado)
    ax1.add_patch(plt.Rectangle((c, GRUPO // lado - 1 - r), 0.86, 0.86, color=mapa[cores_grid[i]]))
ax1.set_xlim(-0.5, lado + 0.5); ax1.set_ylim(-0.5, GRUPO // lado + 0.5)
ax1.set_aspect('equal'); ax1.axis('off')
ax1.set_title(f'De {mil(GRUPO)} pessoas sem sintomas, {vir(verdadeiros + falsos, 0)} testam positivo\n'
              f'{vir(falsos, 0)} são falsos positivos (vermelho) e {vir(verdadeiros, 0)} é doente (verde): '
              f'{vir(p_errado * 100, 0)}% dos positivos estão errados', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-138-grafico-01-mil-pessoas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · As respostas dos médicos
fig2, ax2 = plt.subplots(figsize=(12, 8))
rot = ['acertaram\n(cerca de 2%)', 'responderam\n95%', 'outras\nrespostas']
v14 = np.array([ACERTOS_2014, RESP_95_2014, N_2014 - ACERTOS_2014 - RESP_95_2014]) / N_2014 * 100
v78 = np.array([ACERTOS_1978, RESP_95_1978, N_1978 - ACERTOS_1978 - RESP_95_1978]) / N_1978 * 100
x = np.arange(3)
ax2.bar(x - 0.2, v78, width=0.4, color=CINZA, alpha=0.85, label='1978 · Casscells et al. · N = 60')
ax2.bar(x + 0.2, v14, width=0.4, color=[VERDE, VERMELHO, DOURADO], alpha=0.9, label='2014 · Manrai et al. · N = 61')
for i in range(3):
    ax2.text(i - 0.2, v78[i] + 1, f"{vir(v78[i], 0)}%", ha='center', fontsize=13, color='#444')
    ax2.text(i + 0.2, v14[i] + 1, f"{vir(v14[i], 0)}%", ha='center', fontsize=14, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(rot, fontsize=12)
ax2.set_ylim(0, 60)
ax2.set_ylabel('Médicos, residentes e estudantes (%)', fontsize=12)
ax2.set_title('Três em cada quatro erraram, e a resposta mais comum foi o oposto da certa\n'
              'a mesma pergunta, em hospitais de Boston, com 36 anos de distância', fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=12)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-138-grafico-02-os-medicos.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · A curva de Bayes
fig3, ax3 = plt.subplots(figsize=(12, 8))
ax3.plot(prev * 100, curva_95 * 100, color=VERMELHO, lw=3.5, label='exame do estudo · 5% de falso positivo')
ax3.plot(prev * 100, curva_99 * 100, color=DOURADO, lw=2.5, label='1% de falso positivo')
ax3.plot(prev * 100, curva_999 * 100, color=VERDE, lw=2.5, label='0,1% de falso positivo')
ax3.set_xscale('log')
ax3.axhline(50, color='#333', lw=1, ls='--')
ax3.scatter([PREVALENCIA * 100], [p_errado * 100], s=160, color=VERMELHO, zorder=5, edgecolor='#333')
ax3.annotate(f'a pergunta do estudo\n1 em 1.000 → {vir(p_errado * 100, 0)}% errados',
             xy=(PREVALENCIA * 100, p_errado * 100), xytext=(0.03, 70), fontsize=12, fontweight='bold', color=VERMELHO,
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.5))
ax3.scatter([virada * 100], [50], s=120, color='#333', zorder=5)
ax3.annotate(f'a virada: {vir(virada * 100, 1)}%\nacima disso, o positivo\ntem mais chance de estar certo',
             xy=(virada * 100, 50), xytext=(8, 72), fontsize=11, color='#333',
             arrowprops=dict(arrowstyle='->', color='#333', lw=1.2))
ax3.set_xticks([0.01, 0.1, 1, 10, 50]); ax3.set_xticklabels(['0,01%', '0,1%', '1%', '10%', '50%'], fontsize=12)
ax3.set_xlabel('Taxa-base: proporção de doentes entre quem faz o exame (escala log)', fontsize=12)
ax3.set_ylabel('Chance de o positivo estar errado (%)', fontsize=12)
ax3.set_ylim(0, 104)
ax3.set_title('O mesmo exame erra 98% ou 10%, dependendo de quem o faz\n'
              'P(errado | positivo) = (1 − esp)(1 − p) / [sens · p + (1 − esp)(1 − p)]', fontsize=14, pad=16)
ax3.legend(frameon=False, fontsize=11, loc='lower left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-138-grafico-03-a-curva.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · Quatro exames de verdade
fig4, ax4 = plt.subplots(figsize=(12, 8))
nomes = list(MORGAN.keys())
est = np.array([v[0] for v in MORGAN.values()]) * 100
lo = np.array([v[1] for v in MORGAN.values()]) * 100
hi = np.array([v[2] for v in MORGAN.values()]) * 100
y = np.arange(len(nomes))
ax4.barh(y, hi - lo, left=lo, height=0.45, color=VERDE, alpha=0.85, label='faixa da evidência')
ax4.scatter(est, y, s=220, color=VERMELHO, zorder=5, edgecolor='white', lw=1.5, label='estimativa mediana dos profissionais')
for i in range(len(nomes)):
    ax4.text(est[i] + 1.5, y[i], f"{vir(est[i], 0)}%", va='center', fontsize=13, fontweight='bold', color=VERMELHO)
    ax4.text(lo[i] - 1.5 if lo[i] > 8 else hi[i] + 1.5, y[i] - 0.34,
             f"{vir(lo[i], 0)}% a {vir(hi[i], 0)}%", va='center', ha='right' if lo[i] > 8 else 'left', fontsize=11, color=VERDE)
ax4.set_yticks(y); ax4.set_yticklabels(nomes, fontsize=12)
ax4.invert_yaxis()
ax4.set_xlim(0, 104)
ax4.set_xlabel('Chance de a pessoa ter a doença depois de um exame positivo (%)', fontsize=12)
ax4.set_title('Fora do laboratório, o erro é o mesmo\n553 profissionais, quatro exames reais: a estimativa fica longe da evidência',
              fontsize=14, pad=16)
ax4.legend(frameon=False, fontsize=11, loc='lower right')
plt.figtext(0.5, 0.01, 'Fonte: Morgan et al. (2021), JAMA Internal Medicine 181(6), N = 553 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-138-grafico-04-quatro-exames.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Na pergunta do estudo, 98% dos exames positivos estão errados. Três em cada quatro médicos
erraram a conta, e a resposta mais comum foi 95%, quase o oposto da certa.**

O exame não é ruim. Cinco por cento de falso positivo é um exame bom. O problema é a
taxa-base. Quando 1 pessoa em 1.000 tem a doença, o exame procura uma agulha num palheiro
de 999 sadios, e 5% do palheiro é mais do que a agulha.

O mesmo exame, feito em quem tem sintomas, erra 10%. A diferença não está no exame. Está
em quem o faz.

Fora do laboratório, o erro se repete. Em 2021, 553 profissionais estimaram a chance de
doença depois de uma mamografia de rastreio positiva em 50%. A evidência diz de 3% a 9%.

Se você chutou 95% no começo deste texto, está na companhia da maioria dos médicos de
Boston. Se chutou perto de 2%, pode cobrar o resultado do seu próximo exame com outra
pergunta na mão.

*Da última vez que um exame seu deu positivo, alguém perguntou quantas pessoas como você
fazem esse exame?*

---

### ⚠️ Limitações do Modelo

- As duas amostras de médicos são pequenas (60 e 61 pessoas) e de hospitais de Boston.
  Os intervalos de Jeffreys estão no notebook e são largos.
- A pergunta do estudo supõe sensibilidade de 100% e não diz o que é "falso positivo" com
  precisão. A conta com sensibilidade de 90% dá 1,8% em vez de 2,0%; o resultado não muda.
- A curva da camada 3 é matemática, não dado. Ela vale para qualquer exame com aquela
  especificidade, mas a taxa-base de cada situação real precisa ser conhecida.
- Morgan et al. 2021 mediram estimativas em cenários escritos, não decisões reais.
- Nenhum dos dois estudos foi feito no Brasil.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*